In [2]:
import sqlite3
import pandas as pd
from bokeh.plotting import figure, show, output_notebook
from bokeh.models import ColumnDataSource, HoverTool

output_notebook()

conn = sqlite3.connect('flight_delays.db')

Loading BokehJS ...

In [3]:
test_query = """
    SELECT Reporting_Airline, Flight_Number_Reporting_Airline, Origin, Dest, Distance, ArrDelayMinutes 
    FROM flight_info 
    WHERE ArrDelayMinutes IS NOT NULL AND Distance IS NOT NULL
    ORDER BY RANDOM()
    LIMIT 5000
"""
df = pd.read_sql_query(test_query, conn)
conn.close()

print(f"Loaded {len(df)} rows for Bokeh experiment!")

Loaded 5000 rows for Bokeh experiment!


In [4]:
from bokeh.models import CategoricalColorMapper, Select
from bokeh.palettes import Category10
from bokeh.transform import factor_cmap
from bokeh.layouts import column, row
import itertools

carriers = df['Reporting_Airline'].unique().tolist()
color_mapper = factor_cmap('Reporting_Airline', palette=Category10[10], factors=carriers)

source = ColumnDataSource(df)

hover = HoverTool(tooltips="""
    <div style="background: #333; color: #fff; padding: 10px; border-radius: 5px;">
        <strong>Carrier:</strong> @Reporting_Airline (@Flight_Number_Reporting_Airline)<br>
        <strong>Route:</strong> @Origin ➔ @Dest<br>
        <strong>Distance:</strong> @Distance miles<br>
        <strong>Arrival Delay:</strong> <span style="color: #ff6b6b;">@ArrDelayMinutes mins</span>
    </div>
""")

p = figure(
    title="US Flight Delays: Distance vs. Arrival Delay",
    x_axis_label="Distance (miles)",
    y_axis_label="Arrival Delay (minutes)",
    tools=[hover, "pan", "wheel_zoom", "reset", "save"],
    width=900,
    height=550,
    sizing_mode="scale_width"
)

p.background_fill_color = "#f9f9f9"
p.grid.grid_line_color = "#e0e0e0"
p.grid.grid_line_dash = [6, 4]

carriers = sorted(df['Reporting_Airline'].unique().tolist())
palette = Category10[10]
color_cycle = itertools.cycle(palette)

for carrier, color in zip(carriers, color_cycle):
    # Filter data per airline for individual rendering
    df_carrier = df[df['Reporting_Airline'] == carrier]
    source_carrier = ColumnDataSource(df_carrier)
    
    p.scatter(
        x='Distance', 
        y='ArrDelayMinutes', 
        source=source_carrier,
        size=8,
        color=color,
        alpha=0.7,
        legend_label=carrier,
        hover_color="black",
        hover_alpha=1.0
    )

p.legend.location = "top_left"
p.legend.click_policy = "hide"  # Now clicking an airline will cleanly hide just that airline!
p.title.text_font_size = "14pt"

show(p)

In [5]:
from bokeh.models import CategoricalColorMapper, Select, LassoSelectTool, BoxSelectTool
from bokeh.palettes import Category10
from bokeh.transform import factor_cmap
from bokeh.layouts import column, row
import itertools
import pandas as pd

conn = sqlite3.connect('flight_delays.db')
query = """
    SELECT Reporting_Airline, Flight_Number_Reporting_Airline, Origin, Dest, Distance, DepDelayMinutes, ArrDelayMinutes 
    FROM flight_info 
    WHERE ArrDelayMinutes IS NOT NULL AND Distance IS NOT NULL AND DepDelayMinutes IS NOT NULL
    ORDER BY RANDOM()
    LIMIT 3000
"""
df = pd.read_sql_query(query, conn)
conn.close()

source = ColumnDataSource(df)

tools = ["pan", "wheel_zoom", "box_select", "lasso_select", "reset", "save"]

p1 = figure(
    title="Distance vs. Arrival Delay",
    x_axis_label="Distance (miles)",
    y_axis_label="Arrival Delay (minutes)",
    tools=tools,
    width=500,
    height=450
)

p2 = figure(
    title="Departure vs. Arrival Delay",
    x_axis_label="Departure Delay (minutes)",
    y_axis_label="Arrival Delay (minutes)",
    tools=tools,
    width=500,
    height=450
)

for p in [p1, p2]:
    p.background_fill_color = "#f9f9f9"
    p.grid.grid_line_color = "#e0e0e0"

carriers = sorted(df['Reporting_Airline'].unique().tolist())
palette = Category10[10]
color_cycle = itertools.cycle(palette)

for carrier, color in zip(carriers, color_cycle):
    pass

p1.scatter(
    x='Distance', y='ArrDelayMinutes', source=source,
    size=7, color="navy", alpha=0.6,
    selection_color="firebrick", nonselection_alpha=0.1
)

p2.scatter(
    x='DepDelayMinutes', y='ArrDelayMinutes', source=source,
    size=7, color="teal", alpha=0.6,
    selection_color="firebrick", nonselection_alpha=0.1
)

layout = row(p1, p2)

show(layout)

In [6]:
kpi_conn = sqlite3.connect("flight_delays.db")

kpi_query = """
    SELECT
        Reporting_Airline,
        COUNT(*) AS FlightCount,
        AVG(CASE WHEN ArrDelayMinutes > 0 THEN ArrDelayMinutes END) AS AvgPositiveArrDelay,
        100.0 * SUM(CASE WHEN ArrDelayMinutes <= 15 THEN 1 ELSE 0 END) / COUNT(*) AS OnTimeRate
    FROM flight_info
    WHERE ArrDelayMinutes IS NOT NULL
    GROUP BY Reporting_Airline
"""
kpi_df = pd.read_sql_query(kpi_query, kpi_conn)
kpi_conn.close()

if kpi_df.empty:
    print("No flight data available for these KPIs.")
else:
    chart_specs = [
        ("AvgPositiveArrDelay", "Average Positive Arrival Delay (minutes)", "#e76f51"),
        ("OnTimeRate", "On-time Arrival Rate (%, delay ≤ 15 min)", "#2a9d8f"),
        ("FlightCount", "Number of Flights", "#457b9d"),
    ]

    charts = []
    for metric, title, color in chart_specs:
        chart_df = kpi_df.sort_values(metric, ascending=False).copy()
        chart_source = ColumnDataSource(chart_df)

        p = figure(
            title=title,
            x_range=chart_df["Reporting_Airline"].tolist(),
            width=900,
            height=320,
            tools="pan,wheel_zoom,box_zoom,reset,save",
            sizing_mode="scale_width",
        )
        p.vbar(
            x="Reporting_Airline",
            top=metric,
            source=chart_source,
            width=0.75,
            color=color,
            alpha=0.85,
        )
        p.add_tools(HoverTool(
            tooltips=[
                ("Airline", "@Reporting_Airline"),
                (title, f"@{metric}{{0.0}}"),
            ]
        ))
        p.xaxis.major_label_orientation = 0.8
        p.xaxis.axis_label = "Reporting Airline"
        p.yaxis.axis_label = title
        p.grid.grid_line_color = "#e0e0e0"
        charts.append(p)

    show(column(*charts))